<a href="https://colab.research.google.com/github/P2786/Image-Labeling/blob/main/image_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-cloud-vision ipywidgets
!jupyter nbextension enable --py widgetsnbextension --sys-prefix

import os
from google.cloud import vision
from google.colab import files
from IPython.display import Image, display
import ipywidgets as widgets
from io import BytesIO
from PIL import Image as PILImage

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [ ]:
SERVICE_ACCOUNT_KEY_FILE = '/content/wise-env-495403-m6-20868796afce.json'


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SERVICE_ACCOUNT_KEY_FILE


if not os.path.exists(SERVICE_ACCOUNT_KEY_FILE):
    print(f"Error: Service account key file '{SERVICE_ACCOUNT_KEY_FILE}' not found. Please upload it to Colab.")
else:
    print(f"Authentication set up using {SERVICE_ACCOUNT_KEY_FILE}")

client = vision.ImageAnnotatorClient()

Authentication set up using /content/wise-env-495403-m6-20868796afce.json


In [ ]:
def detect_labels(image_content=None, image_uri=None):
    """Detects labels in the image using the Vision API."""
    image = vision.Image()
    if image_content:
        image.content = image_content
    elif image_uri:
        image.source.image_uri = image_uri
    else:
        return "No image provided."

    response = client.label_detection(image=image)
    labels = response.label_annotations

    result = []
    for label in labels:
        result.append(f'{label.description} (score: {label.score:.2f})')
    return result


image_url_input = widgets.Text(
    value='',
    placeholder='Enter image URL (e.g., https://example.com/image.jpg)',
    description='Image URL:',
    disabled=False
)

uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.gif',
    multiple=False,  # upload only 1 file
    description='Upload Image'
)

output_image = widgets.Output()
output_labels = widgets.Output()

def on_url_submit(change):
    with output_image:
        output_image.clear_output()
        if image_url_input.value:
            try:
                display(Image(url=image_url_input.value, width=200, height=200))
            except Exception as e:
                print(f"Could not display image from URL: {e}")

    with output_labels:
        output_labels.clear_output()
        if image_url_input.value:
            print("Detecting labels from URL...")
            labels = detect_labels(image_uri=image_url_input.value)
            if labels:
                print("Detected Labels:")
                for label in labels:
                    print(f"- {label}")
            else:
                print("No labels detected or an error occurred.")


def on_upload_change(change):
    with output_image:
        output_image.clear_output()
        if uploader.value:
            uploaded_file = next(iter(uploader.value.values()))
            content = uploaded_file['content']
            try:
                img = PILImage.open(BytesIO(content))
                img.thumbnail((200, 200))
                display(img)
            except Exception as e:
                print(f"Could not display uploaded image: {e}")

    with output_labels:
        output_labels.clear_output()
        if uploader.value:
            uploaded_file = next(iter(uploader.value.values()))
            content = uploaded_file['content']
            print("Detecting labels from uploaded image...")
            labels = detect_labels(image_content=content)
            if labels:
                print("Detected Labels:")
                for label in labels:
                    print(f"- {label}")
            else:
                print("No labels detected or an error occurred.")

image_url_input.observe(on_url_submit, names='value')
uploader.observe(on_upload_change, names='value')

display(widgets.VBox([
    widgets.HTML("<h3>Google Cloud Vision API: Image Labeling</h3>"),
    image_url_input,
    widgets.HTML("<p>-- OR --</p>"),
    uploader,
    output_image,
    output_labels
]))